# Week 8 Lab: CNNs with PyTorch (LeNet-5)

## Foreword
In this lab, we build our first **Convolutional Neural Network (CNN)** using **PyTorch**.
We will implement the classic **LeNet-5** architecture to classify handwritten digits (MNIST).

PyTorch simplifies the process:
*   No more manual backprop (Autograd handles it).
*   Layers are pre-defined (`nn.Conv2d`, `nn.Linear`).

### Step 1: Import Dependencies
We import `torch` and `torchvision`. The latter contains popular datasets (MNIST) and transforms.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

### Step 2: Prepare the Dataset
We use `torchvision` to download and transform the MNIST data.
*   We transform images to Tensors.
*   We normalize them (Mean 0.5, Std 0.5) to help training.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download Training Data (This will create a 'data' folder)
try:
    trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

    testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)
    print("Data loaded successfully.")
except Exception as e:
    print(f"Error loading data: {e}. Please ensure you have internet access.")

### Step 3: Define the LeNet-5 Architecture
We define a class that inherits from `nn.Module`.
LeNet-5 Structure:
1.  **Conv1**: 1 input channel (grayscale) -> 6 output channels (5x5 kernel).
2.  **Pool**: 2x2 Max Pooling.
3.  **Conv2**: 6 input channels -> 16 output channels.
4.  **Pool**: 2x2 Max Pooling.
5.  **FC1** (Fully Connected): Flatten -> 120 neurons.
6.  **FC2**: 84 neurons.
7.  **Output**: 10 neurons (Digits 0-9).

In [ ]:
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # Feature Extractor (Convolutional Layers)
        # TODO: Define Conv2d (1, 6, 5)
        self.conv1 = None
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        
        # Classifier (Fully Connected Layers)
        # Input to FC1 depends on image size. MNIST is 28x28.
        # Conv1(5x5) -> 24x24 -> Pool(2x2) -> 12x12
        # Conv2(5x5) -> 8x8 -> Pool(2x2) -> 4x4
        # So final feature map is 16 channels * 4 * 4 pixels = 256 inputs.
        # TODO: Define Linear (16*4*4, 120)
        self.fc1 = None
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = LeNet()
print(net)


<details>
<summary><strong>Click for Solution</strong></summary>

```python
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # Feature Extractor (Convolutional Layers)
        self.conv1 = nn.Conv2d(1, 6, 5) # In: 1, Out: 6, Kernel: 5x5
        self.pool = nn.MaxPool2d(2, 2)  # Kernel: 2x2, Stride: 2
        self.conv2 = nn.Conv2d(6, 16, 5)
        
        # Classifier (Fully Connected Layers)
        # Input to FC1 depends on image size. MNIST is 28x28.
        # Conv1(5x5) -> 24x24 -> Pool(2x2) -> 12x12
        # Conv2(5x5) -> 8x8 -> Pool(2x2) -> 4x4
        # So final feature map is 16 channels * 4 * 4 pixels = 256 inputs.
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # Flatten all dimensions except batch
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = LeNet()
print(net)```
</details>

### Step 4: Define Loss and Optimizer
We use **CrossEntropyLoss** (standard for classification) and **SGD** (Stochastic Gradient Descent) with momentum.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

### Step 5: Training Loop
We iterate through the dataset multiple times (epochs).
For each batch:
1.  Zero gradients (`optimizer.zero_grad()`).
2.  Forward pass (`net(inputs)`).
3.  Calculate Loss (`criterion(outputs, labels)`).
4.  Backward pass (`loss.backward()`).
5.  Update weights (`optimizer.step()`).

In [ ]:
epochs = 5
loss_history = []

print("Starting Training...")
for epoch in range(epochs):  
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward + Backward + Optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:    # Print every 200 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}')
            loss_history.append(running_loss / 200)
            running_loss = 0.0

print('Finished Training')

### Step 6: Visualize Loss Curve
Let's see how the error decreased over time.

In [ ]:
plt.plot(loss_history)
plt.xlabel('Batch Intervals (x200)')
plt.ylabel('Loss')
plt.show()

### Step 7: Evaluate on Test Data
We check the accuracy on the 10,000 test images.

In [ ]:
correct = 0
total = 0
# No gradient needed for evaluation
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        # The class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.2f} %')